In [1]:
import pandas as pd
import pyomo.environ as pyo
from pyomo.environ import *
from pyomo.environ import  SolverFactory

In [5]:
minerios = ['tipo1','tipo2']
contem = ['cobre','zinco','magnesio']

requisitos = {
    contem[0]:8,
    contem[1]:6,
    contem[2]:5
}

informacoes = {
    minerios[0]:{
        contem[0]:0.2,
        contem[1]:0.2,
        contem[2]:0.15,
        'custo':90
    },
    minerios[1]:{
        contem[0]:0.3,
        contem[1]:0.25,
        contem[2]:0.10,
        'custo':120
    }
}

In [15]:
model = ConcreteModel()

model.minerios = pyo.Set(initialize=minerios)
model.contem = pyo.Set(initialize=contem)
model.x = pyo.Var(model.minerios, bounds=(0,None), domain=pyo.NonNegativeReals)

def obj(model):
    return sum(
        model.x[m] * informacoes[m]['custo'] for m in model.minerios
    )
model.ojb = pyo.Objective(rule=obj, sense=pyo.minimize)

def restricoes(model,c):
    return sum(
        model.x[m] * informacoes[m][c] for m in model.minerios
    ) >= requisitos[c]
model.restricoes = pyo.Constraint(model.contem, rule=restricoes)

In [16]:
model.pprint()

2 Set Declarations
    contem : Size=1, Index=None, Ordered=Insertion
        Key  : Dimen : Domain : Size : Members
        None :     1 :    Any :    3 : {'cobre', 'zinco', 'magnesio'}
    minerios : Size=1, Index=None, Ordered=Insertion
        Key  : Dimen : Domain : Size : Members
        None :     1 :    Any :    2 : {'tipo1', 'tipo2'}

1 Var Declarations
    x : Size=2, Index=minerios
        Key   : Lower : Value : Upper : Fixed : Stale : Domain
        tipo1 :     0 :  None :  None : False :  True : NonNegativeReals
        tipo2 :     0 :  None :  None : False :  True : NonNegativeReals

1 Objective Declarations
    ojb : Size=1, Index=None, Active=True
        Key  : Active : Sense    : Expression
        None :   True : minimize : 90*x[tipo1] + 120*x[tipo2]

1 Constraint Declarations
    restricoes : Size=3, Index=contem, Active=True
        Key      : Lower : Body                         : Upper : Active
           cobre :   8.0 :  0.2*x[tipo1] + 0.3*x[tipo2] :  +Inf :   

In [17]:
opt = SolverFactory('cplex')
res = opt.solve(model, tee=True)


Welcome to IBM(R) ILOG(R) CPLEX(R) Interactive Optimizer 22.1.1.0
  with Simplex, Mixed Integer & Barrier Optimizers
5725-A06 5725-A29 5724-Y48 5724-Y49 5724-Y54 5724-Y55 5655-Y21
Copyright IBM Corp. 1988, 2022.  All Rights Reserved.

Type 'help' for a list of available commands.
Type 'help' followed by a command name for more
information on commands.

CPLEX> Logfile 'cplex.log' closed.
Logfile 'C:\Users\DECIV\AppData\Local\Temp\tmppqcyv2i5.cplex.log' open.
CPLEX> Problem 'C:\Users\DECIV\AppData\Local\Temp\tmp132pk7jt.pyomo.lp' read.
Read time = 0.00 sec. (0.00 ticks)
CPLEX> Problem name         : C:\Users\DECIV\AppData\Local\Temp\tmp132pk7jt.pyomo.lp
Objective sense      : Minimize
Variables            :       2
Objective nonzeros   :       2
Linear constraints   :       3  [Greater: 3]
  Nonzeros           :       6
  RHS nonzeros       :       3

Variables            : Min LB: 0.000000         Max UB: all infinite   
Objective nonzeros   : Min   : 90.00000         Max   : 120.0000 

In [19]:
for m in model.x:
    print(f"Quantidade de {m} = {pyo.value(model.x[m]):.2f}")

Quantidade de tipo1 = 28.00
Quantidade de tipo2 = 8.00
